# Análise Exploratória: Insurance × Age

## Objetivo Geral
Investigar como a **idade** (`Age`) influencia o **valor gasto com seguro** (`Insurance`) pelos indivíduos do dataset. A análise agrupa idades em faixas etárias para revelar padrões de comportamento financeiro ao longo das diferentes fases da vida adulta.

### Perguntas que este notebook responde:
- Pessoas mais velhas gastam mais com seguro?
- Em qual faixa etária o gasto com seguro é mais variável?
- O comprometimento de renda com seguro aumenta com a idade?

### Estrutura do notebook:
| Seção | Descrição |
|---|---|
| 0 — Setup | Importação de bibliotecas, carregamento e criação das faixas etárias |
| 1 | Distribuição de registros por faixa etária |
| 2 | Box Plot — Distribuição do Insurance por faixa etária |
| 3 | Violin Plot — Densidade por faixa etária |
| 4 | Histograma por faixa etária |
| 5 | Estatísticas resumidas (Média, Mediana, Desvio Padrão) |
| 6 | Média do seguro por idade (linha contínua) |
| 7 | Conclusões |


---
## Seção 0 — Setup e Carregamento dos Dados

**Objetivo:** Importar bibliotecas, carregar o dataset e **criar a variável `Age_Group`**, que agrupa as idades contínuas em 5 faixas etárias discretas.

**Por que criar faixas etárias?**  
A idade é uma variável contínua com muitos valores únicos (18 a 64 = 47 valores distintos). Agrupar em faixas reduz o ruído, facilita a comparação entre grupos demograficamente significativos e torna os gráficos legíveis.  
As faixas escolhidas (`18-25`, `26-35`, `36-45`, `46-55`, `56-64`) refletem fases típicas da vida financeira: início de carreira, consolidação, maturidade, pré-aposentadoria.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_3.csv')

# Criar faixas etárias com pd.cut
# bins: limites de cada faixa | labels: nome de cada faixa | right=True: o limite superior é incluído
bins = [18, 25, 35, 45, 55, 64]
labels = ['18-25', '26-35', '36-45', '46-55', '56-64']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

# Visão geral do dataset
print(f'Shape: {df.shape}')
print(f"\nDistribuição de Age_Group:\n{df['Age_Group'].value_counts().sort_index()}")
print(f"\nEstatísticas de Insurance por Age_Group:")
print(df.groupby('Age_Group', observed=True)['Insurance'].describe().round(2))

---
## Seção 1 — Distribuição de Registros por Faixa Etária

**Objetivo:** Verificar se as faixas etárias têm tamanhos comparáveis antes de realizar qualquer análise.

**Por que isso é necessário?**  
Se uma faixa tiver muito mais registros que as outras, os resultados para ela serão mais confiáveis estatisticamente.  
Por outro lado, se uma faixa tiver pouquíssimos registros (ex: menos de 50), as análises de distribuição seguintes para aquela faixa devem ser interpretadas com cautela, pois qualquer padrão pode ser ruído amostral e não um padrão real.

In [ ]:
contagem = df['Age_Group'].value_counts().sort_index().reset_index()
contagem.columns = ['Age_Group', 'Count']
contagem['Age_Group'] = contagem['Age_Group'].astype(str)

fig = px.bar(
    contagem,
    x='Age_Group', y='Count',
    color='Age_Group',
    text='Count',
    title='Quantidade de Registros por Faixa Etária',
    labels={'Age_Group': 'Faixa Etária', 'Count': 'Quantidade'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Resultado esperado: se as faixas forem relativamente balanceadas,
# as comparações entre elas nas seções seguintes serão mais justas.

---
## Seção 2 — Box Plot: Distribuição do Insurance por Faixa Etária

**Objetivo:** Comparar a distribuição central e a dispersão do gasto com seguro entre as faixas etárias de forma compacta.

**Como ler o Box Plot:**
- **Linha do meio** → mediana (valor central da faixa)
- **Borda inferior da caixa** → Q1: 25% dos dados ficam abaixo deste valor
- **Borda superior da caixa** → Q3: 75% dos dados ficam abaixo
- **Bigodes** → extensão dos valores típicos (1,5× o tamanho da caixa)
- **Pontos isolados** → outliers: valores muito fora do padrão da faixa

**O que buscar neste gráfico:**  
Se a caixa e a mediana subirem progressivamente de `18-25` para `56-64`, isso confirma que a idade tem correlação positiva com o gasto em seguro. Se as caixas tiverem tamanhos muito diferentes, indica que a variabilidade do gasto também muda com a idade.

In [ ]:
labels = sorted(df['Age'].unique()) # Define 'labels' with unique, sorted age groups
fig = px.box(
    df, x='Age', y='Insurance',
    color='Age',
    title='Distribuição do Seguro por Faixa Etária',
    labels={'Insurance': 'Seguro (R$)', 'Age': 'Faixa Etária'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points='outliers',                         # Mostra apenas os pontos outliers
    category_orders={'Age': labels}      # Garante ordem crescente das faixas
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

---
## Seção 3 — Violin Plot: Densidade por Faixa Etária

**Objetivo:** Revelar a **forma completa da distribuição** do seguro em cada faixa etária, incluindo bimodalidade e assimetria que o box plot não captura.

**Como ler o Violin Plot:**  
A **largura do violino** em cada altura representa quantas pessoas têm aquele valor de seguro — onde é mais largo, há maior concentração de indivíduos naquele valor.  
O **mini box plot interno** mantém as referências de mediana e quartis para facilitar a leitura.

**O que buscar neste gráfico:**  
Se o violino de uma faixa tiver dois picos (bimodal), pode indicar subgrupos dentro daquela faixa etária com perfis de seguro muito distintos — por exemplo, pessoas saudáveis versus com condições crônicas. Isso é uma hipótese relevante para o modelo de classificação.

In [ ]:
# Create a violin plot for Insurance vs. Age
fig_violin_insurance_age = px.violin(
    df,
    x='Age',
    y='Insurance',
    title='Violin Plot of Insurance by Age',
    labels={'Age': 'Age', 'Insurance': 'Insurance Cost'}
)
fig_violin_insurance_age.show()

---
## Seção 4 — Histograma por Faixa Etária

**Objetivo:** Visualizar **como os valores de seguro se distribuem em faixas de preço** dentro de cada grupo etário, com painéis separados por faixa.

**Por que Histograma além do Violin?**  
O histograma é mais preciso para identificar onde exatamente estão os picos de concentração dentro de cada faixa de valor. Enquanto o violin mostra a "forma geral", o histograma permite ver: a maioria da faixa `56-64` gasta entre R$X e R$Y de seguro?  

O uso de `facet_col` cria um painel independente por faixa etária, permitindo comparação direta sem sobreposição.

In [ ]:
# Create 'Age_Group' column by categorizing 'Age'
bins = [0, 18, 25, 35, 45, 55, 65, df['Age'].max() + 1]
bin_labels = ['<18', '18-25', '26-35', '36-45', '46-55', '56-65'] # Removed '>65' label
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=bin_labels, right=False, duplicates='drop')

# Sort the labels for consistent ordering in plots
labels = sorted(df['Age_Group'].unique().astype(str))

fig = px.histogram(
    df, x='Insurance', color='Age_Group',
    facet_col='Age_Group',               # Um painel por faixa etária
    nbins=50,                            # 50 bins: granularidade suficiente sem ruído excessivo
    title='Histograma do Seguro por Faixa Etária',
    labels={'Insurance': 'Seguro (R$)', 'Age_Group': 'Faixa Etária'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    category_orders={'Age_Group': labels} # Garante ordem crescente das faixas etárias
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Observar: a distribuição parece deslocar-se para a direita (valores maiores)
# conforme a faixa etária avança? Isso confirmaria a correlação positiva entre idade e seguro.

---
## Seção 5 — Estatísticas Resumidas: Média, Mediana e Desvio Padrão

**Objetivo:** Quantificar numericamente as diferenças entre faixas etárias e comparar as três métricas principais lado a lado.

**Por que comparar Média e Mediana?**  
Se a média for muito maior que a mediana em uma faixa, significa que **outliers (pessoas com gastos extremos) estão puxando a média para cima**, distorcendo a percepção do gasto típico. A mediana, por ser resistente a outliers, representa melhor o perfil da maioria.

**O que o Desvio Padrão alto revela?**  
Um desvio padrão elevado dentro de uma faixa etária indica que **a idade sozinha não determina o valor do seguro** — outros fatores (renda, número de dependentes, condições de saúde) também são determinantes. Isso é um sinal importante para a engenharia de features do modelo de ML.

In [ ]:
# Calcular estatísticas agregadas por faixa etária
stats = df.groupby('Age_Group', observed=True)['Insurance'].agg(
    Media='mean',
    Mediana='median',
    Desvio_Padrao='std'
).reset_index().round(2)
stats['Age_Group'] = stats['Age_Group'].astype(str)

# Três subgráficos lado a lado: Média | Mediana | Desvio Padrão
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Média do Seguro', 'Mediana do Seguro', 'Desvio Padrão'])

colors = px.colors.qualitative.Bold[:5]

for i, col in enumerate(['Media', 'Mediana', 'Desvio_Padrao'], 1):
    fig.add_trace(
        go.Bar(
            x=stats['Age_Group'], y=stats[col],
            marker_color=colors,
            text=stats[col], textposition='outside',
            showlegend=False
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Estatísticas do Seguro por Faixa Etária',
    plot_bgcolor='white',
    height=450
)
fig.show()

# Tabela completa no console
print(stats.to_string(index=False))

---
## Seção 6 — Média do Seguro por Idade (Gráfico de Linha)

**Objetivo:** Analisar a **evolução granular** do gasto médio com seguro ano a ano, sem o agrupamento em faixas, revelando padrões e anomalias pontuais.

**Por que uma análise por idade individual além das faixas?**  
O agrupamento em faixas etárias (seções anteriores) resume bem o comportamento geral, mas **mascara variações específicas**. Uma anomalia em uma idade isolada (ex: pico inesperado aos 42 anos) seria invisível na faixa `36-45`.  
O gráfico de linha com `Age` no eixo X e a **média do seguro naquela idade exata** revela essa granularidade, permitindo identificar idades críticas ou transições abruptas no padrão de gasto.

In [ ]:
# Calcular média de seguro para cada idade individual (18, 19, 20, ..., 64)
media_por_idade = df.groupby('Age')['Insurance'].mean().reset_index()

fig = px.line(
    media_por_idade,
    x='Age', y='Insurance',
    markers=True,   # Marcadores em cada ponto de idade para facilitar leitura dos valores exatos
    title='Média do Seguro por Idade',
    labels={'Age': 'Idade', 'Insurance': 'Média do Seguro (R$)'}
)
fig.update_traces(
    line_color='#4C72B0',    # Linha azul
    marker_color='#DD8452'   # Marcadores laranja — contraste visual para destacar os pontos
)
fig.update_layout(plot_bgcolor='white')
fig.show()

# O que observar:
# - A linha sobe de forma geral da esquerda para a direita? → correlação positiva confirmada
# - Existem quedas ou picos em idades específicas? → possíveis anomalias ou subgrupos
# - A tendência é linear ou acelerada? → impacta a escolha de algoritmos para o modelo

---
## Seção 7 — Conclusões

**Objetivo:** Consolidar os achados da análise e extrair implicações diretas para o modelo de ML.

Com base na análise acima:

- O valor gasto com **seguro tende a crescer com a idade**, o que é esperado dado o maior risco de saúde em idades mais avançadas.
- O **desvio padrão** é alto em todas as faixas, indicando grande variabilidade mesmo dentro da mesma faixa etária — a idade por si só não determina completamente o gasto com seguro.
- A faixa **56-64** apresenta a maior média e mediana de seguro, confirmando que o comprometimento financeiro com saúde cresce ao longo da vida.
- O gráfico de linha (Seção 6) confirma a **correlação positiva** entre idade e gasto com seguro com granularidade individual.

> **Implicação para o modelo de ML:** A variável `Age` (ou `Age_Group`) tem poder preditivo relevante sobre `Insurance` e deve ser incluída no modelo. Contudo, a alta variância interna sugere que outras variáveis (como renda, número de dependentes ou condições de saúde) combinadas com a idade produzirão um modelo mais preciso. Recomenda-se testar interações entre `Age` e `Income` como feature derivada.